In [1]:
# for working part
import numpy as np
import pandas as pd

In [2]:
master_dataset = pd.read_parquet("E:\\Python New\\revenue-intelligence\\data\\raw_data_files\\master_dataset.parquet")

In [3]:
master_dataset.columns

Index(['order_id', 'product_sku', 'quantity', 'item_price', 'line_item_total',
       'exchange_rate', 'pii_id', 'channel', 'order_date', 'order_status',
       'country', 'state', 'pincode', 'profile', 'profile_details',
       'base_cost', 'lead_source', 'category', 'generic_name'],
      dtype='object')

In [ ]:
# revenue prediction = order_month, order_year, is_weekend, sales_amount(target_column), order_count, aov,
#                       day_of_week, lead_source, profile, channel, state, category, quantity, unique_customers,
#                           previous_revenue, rolling_revenue, moving_average_revenue



# customer segmentation = pii_id, state, profile, profile details, customer_lifetime_value, aov, unique_sku_count,
#                            order_count, delivered_rate, days_since_last_order(not possible), average_basket_size, unique_category_count,
#                              rto_rate, average_monthly_order_frequency, average_days_between_orders


# repeat purchase prediction = pii_id, days_since_last_order, order_count, aov, clv, average_days_between_orders, 
#                               average_monthly_order_frequency, orders_last_30_days(not possible), 
#                                   average_basket_size, repeat_purchase(target_column)


# product recommemdation system = order_id, pii_id, product_sku, category, qty, order_date



In [ ]:
revenue_prediction_dataset = master_dataset[['order_date','line_item_total','exchange_rate',
                                             'lead_source','profile','channel','state','category',
                                             'quantity','pii_id','order_id']]



revenue_prediction_dataset['sales_amt'] = revenue_prediction_dataset['line_item_total'] * revenue_prediction_dataset['exchange_rate']

# as per eda approx 73% was with these and remaining was with others, so we decreased the sources
revenue_prediction_dataset['lead_source'] = np.where(
    revenue_prediction_dataset['lead_source'].isin(['referral','indiamart','special_campaign','website','app']),
    revenue_prediction_dataset['lead_source'],'other_source'
)

# as per eda the contributions are not that decreasing, so we cannot decrease profile
revenue_prediction_dataset['profile'] = revenue_prediction_dataset['profile']

# as per eda the 75% is within 3 channels, so we decreased the channels
revenue_prediction_dataset['channel'] = np.where(
    revenue_prediction_dataset['channel'].isin(['website','B2B','retailer']),
    revenue_prediction_dataset['channel'],'other_channel'
)

# arond 89% sales amount is from these 13 states so remaining merged into other_states
revenue_prediction_dataset['state'] = np.where(revenue_prediction_dataset['state'].isin(['Haryana','Rajasthan','Bihar',
                                                                                         'Kerala','Gujarat','Uttar Pradesh',
                                                                                         'Assam','Karnataka','West Bengal',
                                                                                         'Punjab','Maharashtra','Madhya Pradesh',
                                                                                         'Tamil Nadu','Delhi','Odisha']),
                                                                                         revenue_prediction_dataset['state'],'other_states')

# as per eda, we cannot less the categroy as count and sum contibution is very similar, so keeping as it is 
revenue_prediction_dataset['category'] = revenue_prediction_dataset['category']

# group by
revenue_prediction_dataset = revenue_prediction_dataset.groupby(['order_date','lead_source','profile','channel',
                                                                 'state','category']).agg(
                                                                     order_count=('order_id','count'),
                                                                      sales_amt=('sales_amt','sum'),
                                                                      unique_customers=('pii_id','nunique'),
                                                                      qty=('quantity','sum'))


revenue_prediction_dataset = revenue_prediction_dataset.reset_index().sort_values('order_date')

revenue_prediction_dataset['rolling_revenue_7D'] = revenue_prediction_dataset.groupby(['lead_source','profile','channel',
                                                                                       'state','category'])['sales_amt'].transform(
                                                                                           lambda x: x.shift(1).rolling(7).sum())


revenue_prediction_dataset['moving_avg_7D'] = revenue_prediction_dataset.groupby(['lead_source','profile','channel',
                                                                                       'state','category'])['sales_amt'].transform(
                                                                                           lambda x: x.shift(1).rolling(7).mean())


revenue_prediction_dataset['previous_revenue_1D'] = revenue_prediction_dataset.groupby(['lead_source','profile','channel',
                                                                                       'state','category'])['sales_amt'].shift(1)

revenue_prediction_dataset['previous_revenue_7D'] = revenue_prediction_dataset.groupby(['lead_source','profile','channel',
                                                                                       'state','category'])['sales_amt'].shift(7)

revenue_prediction_dataset['previous_revenue_30D'] = revenue_prediction_dataset.groupby(['lead_source','profile','channel',
                                                                                       'state','category'])['sales_amt'].shift(30)


revenue_prediction_dataset['aov'] = revenue_prediction_dataset['sales_amt'] / revenue_prediction_dataset['order_count']


revenue_prediction_dataset['order_month'] = revenue_prediction_dataset['order_date'].dt.month
revenue_prediction_dataset['order_year'] = revenue_prediction_dataset['order_date'].dt.year
revenue_prediction_dataset['day_of_week'] = revenue_prediction_dataset['order_date'].dt.day_of_week
revenue_prediction_dataset['is_weekend'] = np.where(revenue_prediction_dataset['day_of_week'] >= 5,True,False)



revenue_prediction_dataset.columns



C:\Users\hp5cd\AppData\Local\Temp\ipykernel_16780\3001728946.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  revenue_prediction_dataset['sales_amt'] = revenue_prediction_dataset['line_item_total'] * revenue_prediction_dataset['exchange_rate']
C:\Users\hp5cd\AppData\Local\Temp\ipykernel_16780\3001728946.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  revenue_prediction_dataset['lead_source'] = np.where(
C:\Users\hp5cd\AppData\Local\Temp\ipykernel_16780\3001728946.py:16: SettingWithCopyWarning: 
A va

Index(['order_date', 'lead_source', 'profile', 'channel', 'state', 'category',
       'order_count', 'sales_amt', 'unique_customers', 'qty',
       'rolling_revenue_7D', 'moving_avg_7D', 'previous_revenue_1D',
       'previous_revenue_7D', 'previous_revenue_30D', 'aov', 'order_month',
       'order_year', 'day_of_week', 'is_weekend'],
      dtype='object')

In [7]:
temp = master_dataset.copy()

temp['year_month'] = temp['order_date'].dt.to_period('M')

monthly_orders = (
    temp.groupby(['pii_id', 'year_month'])['order_id']
        .nunique()
        .reset_index(name='monthly_orders')
)

avg_monthly_order_frequency = (
    monthly_orders.groupby('pii_id')['monthly_orders']
        .mean()
        .reset_index(name='average_monthly_order_frequency')
)


temp = master_dataset.sort_values(['pii_id', 'order_date'])

temp['days_between_orders'] = (
    temp.groupby('pii_id')['order_date']
        .diff()
        .dt.days
)

average_days_between_orders = (
    temp.groupby('pii_id')['days_between_orders']
        .mean()
        .reset_index(name='average_days_between_orders')
)

In [8]:

customer_segmentation_dataset = master_dataset.groupby(['pii_id','state','profile']).agg(clv=('line_item_total','sum'),
                                                                                        order_count=('order_id','count'),
                                                                                        unique_sku_purchased=('product_sku','nunique'),
                                                                                        delivered_count=('order_status',lambda x:(x=='delivered').sum()),
                                                                                        last_order_date=('order_date','max'),
                                                                                        unique_category_count=('category','nunique'),
                                                                                        rto_count=('order_status',lambda x:(x=='rto').sum()),
                                                                                        total_qty=('quantity','sum'))


customer_segmentation_dataset['aov'] = customer_segmentation_dataset['clv'] / customer_segmentation_dataset['order_count']

customer_segmentation_dataset['delivered_rate'] = customer_segmentation_dataset['delivered_count'] / customer_segmentation_dataset['order_count']


customer_segmentation_dataset['rto_rate'] = customer_segmentation_dataset['rto_count'] / customer_segmentation_dataset['order_count']

customer_segmentation_dataset['average_basket_size'] = customer_segmentation_dataset['total_qty'] / customer_segmentation_dataset['order_count']


customer_segmentation_dataset = customer_segmentation_dataset.merge(avg_monthly_order_frequency,on='pii_id',how='left')

customer_segmentation_dataset = customer_segmentation_dataset.merge(average_days_between_orders,on='pii_id',how='left')


customer_segmentation_dataset.columns

Index(['pii_id', 'clv', 'order_count', 'unique_sku_purchased',
       'delivered_count', 'last_order_date', 'unique_category_count',
       'rto_count', 'total_qty', 'aov', 'delivered_rate', 'rto_rate',
       'average_basket_size', 'average_monthly_order_frequency',
       'average_days_between_orders'],
      dtype='object')

In [10]:

repeat_purchase_prediction_dataset = customer_segmentation_dataset[['pii_id', 'clv', 'order_count','aov',
                                                                   'average_basket_size', 'average_monthly_order_frequency',
                                                                   'average_days_between_orders']]


repeat_purchase_prediction_dataset['repeat_purchase'] = np.where(repeat_purchase_prediction_dataset['order_count']>1,True,False)

repeat_purchase_prediction_dataset.columns

C:\Users\hp5cd\AppData\Local\Temp\ipykernel_16780\4246033090.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  repeat_purchase_prediction_dataset['repeat_purchase'] = np.where(repeat_purchase_prediction_dataset['order_count']>1,True,False)


Index(['pii_id', 'clv', 'order_count', 'aov', 'average_basket_size',
       'average_monthly_order_frequency', 'average_days_between_orders',
       'repeat_purchase'],
      dtype='object')

In [11]:
product_recomendation_dataset = master_dataset[['order_id','pii_id','product_sku','category','quantity','order_date']]

product_recomendation_dataset.columns

Index(['order_id', 'pii_id', 'product_sku', 'category', 'quantity',
       'order_date'],
      dtype='object')

In [12]:
# finally loading the processed dataset into parquet file


product_recomendation_dataset.to_parquet("E:\\Python New\\revenue-intelligence\\data\\processed_data_files\\product_recomendation_dataset.parquet",
                                         engine="fastparquet",index=False)

repeat_purchase_prediction_dataset.to_parquet("E:\\Python New\\revenue-intelligence\\data\\processed_data_files\\repeat_purchase_prediction_dataset.parquet",
                                              engine="fastparquet",index=False)

customer_segmentation_dataset.to_parquet("E:\\Python New\\revenue-intelligence\\data\\processed_data_files\\customer_segmentation_dataset.parquet",
                                         engine="fastparquet",index=False)

revenue_prediction_dataset.to_parquet("E:\\Python New\\revenue-intelligence\\data\\processed_data_files\\revenue_prediction_dataset.parquet",
                                      engine="fastparquet",index=False)